# H3 Export Demo

This notebook demonstrates TerraFlow's H3-indexed export workflow.

H3 is Uber's hierarchical hexagonal geospatial indexing system. By converting TerraFlow
suitability results to H3 cells, you can:

- Directly visualize results in H3-native tools such as DeckGL and Kepler.gl
- Join with other H3-indexed datasets (land use, population, climate normals)
- Aggregate at multiple resolutions without reprojection

TerraFlow uses the H3 v4 API (`h3.latlng_to_cell`). Install the optional extra with:

```bash
pip install terraflow-agro[h3]
```

In [ ]:
try:
    import h3
except ImportError:
    raise ImportError(
        "h3 is required for this notebook. Install it with: pip install terraflow-agro[h3]"
    )

import pandas as pd
import numpy as np
from terraflow.export import to_h3

# Synthetic features DataFrame mimicking TerraFlow pipeline output
rng = np.random.default_rng(42)
n = 10
features_df = pd.DataFrame({
    "lat": rng.uniform(39.9, 40.1, n),
    "lon": rng.uniform(-100.1, -99.9, n),
    "score": rng.uniform(0.2, 0.9, n),
    "v_index": rng.uniform(0.0, 25.0, n),
    "mean_temp": rng.uniform(10.0, 30.0, n),
    "total_rain": rng.uniform(50.0, 300.0, n),
    "label": rng.choice(["Low", "Medium", "High"], n),
})
print(f"Synthetic features: {len(features_df)} rows")
features_df.head()

In [ ]:
# Convert to H3 cells at resolution 8 (~0.74 km² per cell)
h3_df = to_h3(features_df, resolution=8)
print(f"H3 cells at resolution 8: {len(h3_df)} unique cells from {len(features_df)} input rows")
h3_df

In [ ]:
# Compare at a coarser resolution (resolution 4 ~  1,770 km² per cell)
h3_coarse = to_h3(features_df, resolution=4)
print(f"Resolution 8: {len(h3_df)} cells")
print(f"Resolution 4: {len(h3_coarse)} cells  (coarser — more rows merged per cell)")
h3_coarse

## Visualization with DeckGL / Kepler.gl

H3 cells can be directly consumed by H3-native visualization tools:

- **DeckGL H3HexagonLayer**: pass the `h3_cell` index column directly — no geometry
  conversion needed. Color-map `score` to visualize suitability across the region.
- **Kepler.gl**: import the exported Parquet as a dataset, select the H3 index column,
  and choose "H3" as the layer type.
- **pandas-h3 / h3pandas**: supports `to_h3`, `h3_to_geo_boundary`, and resolution
  changes directly on DataFrames.

Because H3 uses a consistent hexagonal grid, adjacent cells always share edges —
eliminating distortion artifacts common in rectangular pixel grids at high latitudes.

In [ ]:
# CLI equivalent — run this after a successful `terraflow run -c config.yml`
# !terraflow export --format h3 -c config.yml
# Override resolution:
# !terraflow export --format h3 --resolution 4 -c config.yml
print("CLI: terraflow export --format h3 -c config.yml")
print("Output: outputs/runs/<fingerprint>/h3_resolution_8.parquet")